# 🆚 Change Detection (BIT / ChangeOS style) — One-click Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedRafiAhmedGIS/AnyDisasterMapping/blob/main/change_detection_colab.ipynb)
[![GitHub](https://img.shields.io/badge/Repo-AnyDisasterMapping-181717?logo=github)](https://github.com/SyedRafiAhmedGIS/AnyDisasterMapping)
![Task](https://img.shields.io/badge/task-cd-4c8cbf) ![Auto](https://img.shields.io/badge/runs-automatically-3fb950)

Upload a **pre**- and **post**-disaster image pair and get a changed-area map. This notebook runs in two honest modes: **CHECKPOINT** (a real trained `TinyUNetSiam` — train one in 3 minutes in `satellite_finetune_colab.ipynb`, or upload any `.pth` it produced) or **SYNTHETIC-DEMO** (clearly-labeled heuristic on synthetic scenes). For the full BIT / ChangeMamba / ChangeOS models, train with the repo configs — pointers below.

| ▶️ How to run | |
|---|---|
| **Everything automatic** | Click **Runtime ▸ Run all** (Ctrl+F9): setup → demo scene → inference → dashboard → export chain without any further clicks |
| **With your data** | Run the cells once, then click the 📤 upload buttons — the pipeline re-runs automatically after upload |
| **No image at hand?** | The 🛟 fallback generates a clearly-labeled synthetic tsunami scene — the notebook *never blocks or crashes* |

**📤 Upload preview — what to have ready:**

| file | what it is | typical size |
|---|---|---|
| **`pre_tsunami.png`** | earlier-date image of the area | 512–2048 px, ≤ 20 MB |
| **`post_tsunami.png`** | later-date image after the event | 512–2048 px, ≤ 20 MB |
| **`tiny_unet_cd.pth`** | optional checkpoint (from the fine-tune notebook) | any |
| **`no files?`** | click 🛟 — synthetic pre/post pair is generated | — |

> 🔖 Every result is stamped with its mode: **CHECKPOINT** (your `.pth`), **QUICK-TRAIN** (genuinely briefly-trained), or **SYNTHETIC-DEMO** (clearly-labeled heuristic). Nothing is faked.

In [ ]:
#@title ⚙️ SETUP — auto-installs, idempotent, restart-safe {display-mode:"form"}
AUTO_RUN = True  #@param {type:"boolean"}
SHOW_PREVIEW_ON_UPLOAD = True  #@param {type:"boolean"}
import os, sys, glob, json, math, random, subprocess, traceback, datetime, textwrap

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

colab_kit.pip_install("ipywidgets", "pillow", "numpy", "imageio", "scikit-image", "tifffile", "matplotlib")

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageDraw
import ipywidgets as widgets
from IPython.display import display, HTML, Image as IPImage, clear_output
from colab_kit import (make_synthetic, infer, overlay, heatmap, to_u8, stats_for,
                       stylize_poster, read_any, verdict, show_pair, run_inference,
                       run_dashboard, export_results_zip_a, nb_save_upload, nb_apply,
                       nb_upload_pre, nb_upload_post, nb_use_synthetic,
                       _field, _fbm, _stretch, _blur, _median_mask, _terrain, _hsv,
                       segment_heuristic, change_heuristic, _otsu)

random.seed(42); np.random.seed(42)
OUT_DIR = "outputs"; os.makedirs(OUT_DIR, exist_ok=True)
DOMAIN = "tsunami"          # this notebook's disaster domain
TASK   = "cd"            # "seg" | "cd" | "both"
GSD_M  = 10.0               # ground sample distance (m/pixel) for km² stats
NOTEBOOK = "change_detection_colab.ipynb"
STATE = {"domain": DOMAIN, "task": "seg" if TASK != "cd" else "cd",
         "pre": None, "post": None, "gt": None, "mask": None, "source": "none"}
print("✅ Environment ready — notebook:", NOTEBOOK, "| domain:", DOMAIN, "| task:", TASK)

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


In [ ]:
#@title ☁️ MOUNT GOOGLE DRIVE — persistent uploads, checkpoints & results {display-mode:"form"}
MOUNT_DRIVE = True  #@param {type:"boolean"}
DRIVE_PROJECT = "AnyDisasterMapping"  #@param {type:"string"}

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

DRIVE_ROOT, IN_COLAB = colab_kit.mount_and_prepare(DRIVE_PROJECT, MOUNT_DRIVE)

In [ ]:
#@title 📤 UPLOAD — buttons + inline preview, never blocks {display-mode:"form"}
# upload logic lives in colab_kit (single source of truth, unit-tested in CI)
def _save_upload(name, data):
    return colab_kit.nb_save_upload(globals(), name, data)

def _apply(kind, arr):
    return colab_kit.nb_apply(globals(), kind, arr)

def upload_pre(_=None):
    return colab_kit.nb_upload_pre(globals())

def upload_post(_=None):
    return colab_kit.nb_upload_post(globals())

def use_synthetic(_=None):
    return colab_kit.nb_use_synthetic(globals())


SPEC = [['pre_tsunami.png', 'earlier-date image of the area', '512–2048 px, ≤ 20 MB'],
 ['post_tsunami.png', 'later-date image after the event', '512–2048 px, ≤ 20 MB'],
 ['tiny_unet_cd.pth', 'optional checkpoint (from the fine-tune notebook)', 'any'],
 ['no files?', 'click 🛟 — synthetic pre/post pair is generated', '—']]
W = 70
def print_spec():
    print("┌" + "─" * W + "┐")
    for line in textwrap.wrap("📤 WHAT TO UPLOAD — " + NOTEBOOK, W - 2):
        print("│ " + line.ljust(W - 2) + "│")
    for label, desc, size in SPEC:
        wrapped = textwrap.wrap(label + " — " + desc + "  (" + size + ")", W - 6)
        for j, line in enumerate(wrapped):
            print(("│   • " if j == 0 else "│     ") + line.ljust(W - 6) + "│")
    for line in textwrap.wrap(
            "accepted: .png .jpg .jpeg .tif .tiff .h5 | sweet spot: 512–2048 px, ≤ 20 MB each | "
            "GeoTIFF georeference is kept when present", W - 2):
        print("│ " + line.ljust(W - 2) + "│")
    print("└" + "─" * W + "┘")
print_spec()

out_up = widgets.Output(layout={"border": "1px solid #ddd"})
domain_sel = widgets.Dropdown(
    options=["tsunami", "flood", "hurricane", "oil_spill", "glof", "locust"],
    value="tsunami", description="Domain:")
def _sync_dom(*_):
    STATE["domain"] = domain_sel.value
domain_sel.observe(_sync_dom, names="value")
# (task fixed for this notebook)
gsd_box = widgets.FloatText(value=GSD_M, description="GSD m/px:",
                            tooltip="ground sample distance, used for km² statistics")
btn_post = widgets.Button(description="⬆️ Upload POST image", button_style="success", icon="upload")
btn_extra = None

btn_extra = widgets.Button(description="⬆️ Upload .pth checkpoint (optional)",
                           button_style="info", icon="upload")
def upload_ckpt(_=None):
    with out_up:
        clear_output()
        print("📂 choose a TinyUNetSiam .pth (e.g. outputs/tiny_unet_cd.pth from "
              "satellite_finetune_colab) …")
    try:
        for name, data in _gfiles.upload().items():
            p = os.path.join(OUT_DIR, "ckpt_user.pth")
            with open(p, "wb") as f:
                f.write(data)
            STATE["ckpt"] = p
            with out_up:
                print("✅ checkpoint stored →", p, "— RUN now uses CHECKPOINT mode")
    except Exception as e:
        with out_up:
            print("upload failed:", e)
btn_extra.on_click(upload_ckpt)


btn_pre = widgets.Button(description="⬆️ Upload PRE image", button_style="primary", icon="upload")
btn_synth = widgets.Button(description="🛟 Use bundled synthetic sample",
                           button_style="warning", icon="picture")
btns = [btn_pre]
if TASK in ("cd", "both"):
    btns.insert(1, btn_post)
if btn_extra is not None:
    btns.append(btn_extra)
box = widgets.VBox([widgets.HBox(btns), widgets.HBox([btn_synth, gsd_box]), out_up])






btn_pre.on_click(upload_pre)
if TASK in ("cd", "both"):
    btn_post.on_click(upload_post)
btn_synth.on_click(use_synthetic)
display(box)

if AUTO_RUN and STATE["pre"] is None:
    print("⚡ AUTO_RUN: generating bundled synthetic sample — upload later to re-run automatically")
    use_synthetic()

In [ ]:
#@title 🧠 MODEL — siamese change-detection network (TinyUNetSiam) {display-mode:"form"}
print("torch:", end=" ")

import torch
import torch.nn as nn

class TinyUNetSiam(nn.Module):
    """6-channel (pre+post) -> 1-channel change logits. 3-level UNet, CPU-friendly."""
    def __init__(self, ch=16):
        super().__init__()
        def blk(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.ReLU(),
                                 nn.Conv2d(o, o, 3, padding=1), nn.ReLU())
        self.e1 = blk(6, ch);    self.p1 = nn.MaxPool2d(2)
        self.e2 = blk(ch, ch*2); self.p2 = nn.MaxPool2d(2)
        self.mid = blk(ch*2, ch*4)
        self.u2 = nn.ConvTranspose2d(ch*4, ch*2, 2, stride=2)
        self.d2 = blk(ch*4, ch*2)
        self.u1 = nn.ConvTranspose2d(ch*2, ch, 2, stride=2)
        self.d1 = blk(ch*2, ch)
        self.out = nn.Conv2d(ch, 1, 1)
    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.p1(e1))
        m  = self.mid(self.p2(e2))
        d2 = self.d2(torch.cat([self.u2(m), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

print("cuda available:", torch.cuda.is_available(),
      "— full BIT/ChangeMamba training wants a GPU; TinyUNetSiam is CPU-happy")

In [ ]:
#@title ▶️ RUN — labeled inference + results dashboard {display-mode:"form"}
MODE_TAG = "SYNTHETIC-DEMO"

def try_checkpoint():
    """CHECKPOINT mode: load TinyUNetSiam weights (uploaded or trained in-session)."""
    p = STATE.get("ckpt")
    if p is None:
        cands = sorted(glob.glob(os.path.join(OUT_DIR, "*.pth")))
        p = cands[-1] if cands else None
    if p is None:
        return None
    try:
        sd = torch.load(p, map_location="cpu")
        net = TinyUNetSiam()
        net.load_state_dict(sd)
        net.eval()
        def _fn(pre, post):
            x = torch.from_numpy(np.concatenate([pre, post], -1).transpose(2, 0, 1)).float()[None]
            with torch.no_grad():
                logits = net(x)[0, 0].numpy()
            return _median_mask(logits > 0.0, 5)
        print("✅ CHECKPOINT mode —", os.path.basename(p))
        return _fn, "CHECKPOINT"
    except Exception as e:
        print("⚠️ checkpoint unusable (", e, ") → SYNTHETIC-DEMO heuristic")
        return None

def run_inference():
    global MODE_TAG
    pre, post = STATE.get("pre"), STATE.get("post")
    if pre is None or (STATE["task"] == "cd" and post is None):
        print("🛟 no usable pair in STATE — falling back to bundled synthetic sample")
        pre, post, gt = make_synthetic(STATE["domain"], 512, 512, seed=7)
        STATE.update(pre=pre, post=post, gt=gt, source="synthetic")
    ck = try_checkpoint()
    if ck is not None:
        fn, MODE_TAG = ck
        mask = fn(STATE["pre"], STATE["post"])
    else:
        mask, MODE_TAG = infer(STATE["domain"], STATE["task"], STATE["pre"], STATE["post"])
    STATE["mask"] = mask
    STATE["mode_tag"] = MODE_TAG
    return mask

def run_pipeline():
    mask = run_inference()
    gsd = STATE.get("gsd", GSD_M)
    st = stats_for(mask, gsd)
    st["mode"] = STATE.get("mode_tag", MODE_TAG)
    st["domain"] = STATE["domain"]
    st["task"] = STATE["task"]
    st["timestamp"] = datetime.datetime.now().isoformat(timespec="seconds")
    pre, post, tag = STATE["pre"], STATE["post"], st["mode"]
    if STATE["task"] == "cd":
        panels = [pre, post, heatmap(mask), overlay(post, mask)]
        titles = ["PRE", "POST", "change heatmap", "changed-area overlay"]
    else:
        panels = [post, heatmap(mask), overlay(post, mask)]
        titles = ["input image", "heatmap", "overlay"]
    fig, ax = plt.subplots(1, len(panels), figsize=(4.0 * len(panels), 4.6))
    for a, im, t in zip(np.atleast_1d(ax), panels, titles):
        a.imshow(np.clip(im, 0, 1)); a.set_title(t, fontsize=10); a.axis("off")
    fig.suptitle(f"{NOTEBOOK} — mode: {tag}", fontsize=12, fontweight="bold")
    dash = os.path.join(OUT_DIR, "dashboard_" + STATE["domain"] + ".png")
    fig.savefig(dash, dpi=110, bbox_inches="tight"); plt.show()
    Image.fromarray(to_u8(post)).save(os.path.join(OUT_DIR, "post_" + STATE["domain"] + ".png"))
    Image.fromarray(to_u8(overlay(post, mask))).save(
        os.path.join(OUT_DIR, "overlay_" + STATE["domain"] + ".png"))
    np.save(os.path.join(OUT_DIR, "mask_" + STATE["domain"] + ".npy"), mask)
    Image.fromarray((mask * 255).astype(np.uint8)).save(
        os.path.join(OUT_DIR, "mask_" + STATE["domain"] + ".png"))
    with open(os.path.join(OUT_DIR, "stats.json"), "w") as f:
        json.dump(st, f, indent=2)
    print("┌" + "─" * 46 + "┐")
    print(("│ 📊 RESULT — mode: " + tag).ljust(47) + "│")
    print(("│   changed: " + str(st["changed_pct"]) + " % of scene").ljust(47) + "│")
    print(("│   area: " + str(st["km2"]) + " km² (GSD " + str(gsd) + " m/px)").ljust(47) + "│")
    if STATE.get("gt") is not None:
        gt = STATE["gt"]
        iou = float((mask & gt).sum()) / max(1, float((mask | gt).sum()))
        print(("│   IoU vs synthetic ground truth: " + f"{iou:.3f}").ljust(47) + "│")
    print(("│   saved: " + dash).ljust(47) + "│")
    print("└" + "─" * 46 + "┘")
    if AUTO_RUN and "run_export" in globals():
        run_export()

run_pipeline() if AUTO_RUN else print("ready — call run_pipeline() after uploading")

In [ ]:
#@title 📦 EXPORT — results zip + optional Drive copy + auto-download {{display-mode:"form"}}
USE_DRIVE = False  #@param {{type:"boolean"}}

def run_export():
    return colab_kit.export_results_zip_a(globals(), use_drive=USE_DRIVE)

run_export() if AUTO_RUN else print("call run_export() when ready")

## 🔎 Where the real BIT / ChangeOS models live

| what | where |
|---|---|
| Training entry point | `train.py` + `configs/<domain>/<dataset>/*.yaml` |
| Change-family models | `src/models/` — BIT, ChangeMamba (ChangeOS), SpectralGPT+ … |
| Data prep per domain | `scripts/data_prep/<domain>/README.md` |
| Quick fine-tune on your own pairs | open **`satellite_finetune_colab.ipynb`** → upload the resulting `tiny_unet_cd.pth` here |
| Full paper context | see the review paper linked in the main README |

> 💡 GPU runtime is recommended for the big models; this notebook's TinyUNet mode is intentionally light enough for CPU.